In [1]:
import json
import time

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder

from benchmark.config import DATA_PROCESSED, DATASETS, MODELS_DIR, RESULTS_BENCHMARK
from benchmark.metrics import (
    evaluate_classification,
    plot_confusion_matrix,
    plot_roc_curve,
    save_classification_report,
)
from benchmark.models.xrfm_model import make_processor
from benchmark.models.xrfm_model import fit_best_xrfm, fit_xrfm_cross_validation

data_name = "sleep_health"
cfg = DATASETS[data_name]
processed = DATA_PROCESSED / data_name
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/home/shado/college_project/comp9417/9417GroupProject/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_df = pd.read_csv(processed / "train.csv")
train_df.head()

,age,gender,occupation,bmi,country,sleep_duration_hrs,sleep_quality_score,rem_percentage,deep_sleep_percentage,sleep_latency_mins,...,shift_work,room_temperature_celsius,weekend_sleep_diff_hrs,season,day_type,sleep_disorder_risk,rem_hrs,deep_sleep_hrs,light_sleep_hrs,wake_rate
0,23,Male,Software Engineer,29.4,USA,6.67,4.9,17.8,24.1,18,...,0,21.1,0.65,Winter,Weekend,Healthy,1.18726,1.60747,3.87527,0.599700
1,45,Female,Driver,25.7,South Korea,5.98,5.2,21.6,19.5,28,...,0,19.6,0.49,Spring,Weekday,Healthy,1.29168,1.16610,3.52222,0.334448
2,46,Female,Sales,26.7,USA,5.67,4.9,25.6,17.6,10,...,0,25.5,1.61,Winter,Weekday,Healthy,1.45152,0.99792,3.22056,0.176367
3,51,Male,Manager,30.8,USA,6.38,5.3,16.8,13.8,20,...,0,23.8,0.76,Winter,Weekday,Mild,1.07184,0.88044,4.42772,0.313480
4,20,Male,Retired,26.7,USA,9.33,7.1,23.8,23.2,25,...,0,23.8,1.90,Spring,Weekend,Healthy,2.22054,2.16456,4.94490,0.107181


In [3]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 33 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   age                          16000 non-null  int64  
 1   gender                       16000 non-null  object 
 2   occupation                   16000 non-null  object 
 3   bmi                          16000 non-null  float64
 4   country                      16000 non-null  object 
 5   sleep_duration_hrs           16000 non-null  float64
 6   sleep_quality_score          16000 non-null  float64
 7   rem_percentage               16000 non-null  float64
 8   deep_sleep_percentage        16000 non-null  float64
 9   sleep_latency_mins           16000 non-null  int64  
 10  wake_episodes_per_night      16000 non-null  int64  
 11  caffeine_mg_before_bed       16000 non-null  int64  
 12  alcohol_units_before_bed     16000 non-null  float64
 13  screen_time_befo

In [4]:
test_df = pd.read_csv(processed / "test.csv")
test_df.head()

,age,gender,occupation,bmi,country,sleep_duration_hrs,sleep_quality_score,rem_percentage,deep_sleep_percentage,sleep_latency_mins,...,shift_work,room_temperature_celsius,weekend_sleep_diff_hrs,season,day_type,sleep_disorder_risk,rem_hrs,deep_sleep_hrs,light_sleep_hrs,wake_rate
0,29,Male,Lawyer,30.7,Brazil,6.00,4.8,21.2,24.0,18,...,0,24.1,1.17,Spring,Weekday,Mild,1.27200,1.44000,3.28800,1.000000
1,60,Female,Freelancer,26.3,Japan,7.37,6.8,24.0,17.7,19,...,0,23.3,1.48,Winter,Weekday,Healthy,1.76880,1.30449,4.29671,0.407056
2,23,Male,Manager,24.3,India,7.46,6.6,21.6,27.4,11,...,0,24.5,0.91,Spring,Weekday,Healthy,1.61136,2.04404,3.80460,0.402145
3,26,Male,Doctor,26.6,India,6.34,3.4,15.9,12.7,41,...,0,18.9,0.40,Summer,Weekday,Mild,1.00806,0.80518,4.52676,0.473186
4,28,Male,Freelancer,22.2,UK,5.14,3.1,25.5,19.8,32,...,0,20.7,0.26,Spring,Weekday,Moderate,1.31070,1.01772,2.81158,0.583658


In [5]:
with open(processed / "metadata.json") as f:
    metadata = json.load(f)

print(metadata)

{'num_cols': ['age', 'bmi', 'sleep_duration_hrs', 'sleep_quality_score', 'rem_percentage', 'deep_sleep_percentage', 'sleep_latency_mins', 'wake_episodes_per_night', 'caffeine_mg_before_bed', 'alcohol_units_before_bed', 'screen_time_before_bed_mins', 'exercise_day', 'steps_that_day', 'nap_duration_mins', 'stress_score', 'work_hours_that_day', 'heart_rate_resting_bpm', 'room_temperature_celsius', 'weekend_sleep_diff_hrs', 'rem_hrs', 'deep_sleep_hrs', 'light_sleep_hrs', 'wake_rate'], 'cat_cols': ['gender', 'occupation', 'country', 'chronotype', 'mental_health_condition', 'season', 'day_type'], 'bin_cols': ['sleep_aid_used', 'shift_work'], 'target_col': 'sleep_disorder_risk'}


In [6]:
le = LabelEncoder()
X = train_df.drop(columns=metadata["target_col"])
y = le.fit_transform(train_df[metadata["target_col"]]).astype(np.int32)

print("Classes:", le.classes_)

processor = make_processor(metadata)
n_num_cols = len(metadata["num_cols"])

Classes: ['Healthy' 'Mild' 'Moderate' 'Severe']


In [7]:
best_trial = fit_xrfm_cross_validation(
    X, y, processor, n_num_cols, task=cfg["task"], device=device
)

print(f"Best params: {best_trial.params}")
print(f"Best macro F1: {best_trial.value}")

[I 2026-04-22 21:46:45,181] A new study created in memory with name: no-name-2b874564-7b49-4e30-967d-9e5c82dc8d6e


None
Fitting xRFM with 1 trees and 0 iterations per tree


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]

Fitting RFM with ntrain: 12800, d: 70, and nval: 3200
Using cheap batch size
Optimal M batch size: 2661
Time taken for round 0: 0.5671148300170898 seconds
Using cheap batch size
Optimal M batch size: 2661
Time taken for round 1: 0.259413480758667 seconds
Using cheap batch size
Optimal M batch size: 2661
Time taken for round 2: 0.258436918258667 seconds
Using cheap batch size
Optimal M batch size: 2661
Time taken for round 3: 0.2625851631164551 seconds
Using cheap batch size
Optimal M batch size: 2661
Time taken for round 4: 0.27328038215637207 seconds


Building trees:   0%|          | 0/1 [00:01<?, ?it/s]


...

Tree has no split, stopping training
Using hard routing for tree prediction
None
Fitting xRFM with 1 trees and 0 iterations per tree



Building trees:   0%|          | 0/1 [00:00<?, ?it/s]

Fitting RFM with ntrain: 12800, d: 70, and nval: 3200
Using expensive batch size
Optimal M batch size: 398
Using SVD
Time taken for round 0: 2.1429085731506348 seconds
Using expensive batch size
Optimal M batch size: 398
Using SVD
Time taken for round 1: 5.048088312149048 seconds
Using expensive batch size
Optimal M batch size: 398
Using SVD
Time taken for round 2: 2.08471941947937 seconds
Using expensive batch size
Optimal M batch size: 398
Using SVD
Time taken for round 3: 2.10334849357605 seconds
Using expensive batch size
Optimal M batch size: 398
Using SVD
Time taken for round 4: 2.0889265537261963 seconds


Building trees:   0%|          | 0/1 [00:14<?, ?it/s]
[I 2026-04-22 23:20:27,638] Trial 99 finished with value: 0.7953358242990728 and parameters: {'exponent': 0.9581952058375022, 'kernel_type': 'kpq', 'bandwidth': 31.77149139542352, 'diag': False, 'norm_p': 1.497244663805375, 'reg': 2.0713835983453015e-05}. Best is trial 44 with value: 0.796167149321403.


Tree has no split, stopping training
Using hard routing for tree prediction
Best params: {'exponent': 0.9205640134127895, 'kernel_type': 'kpq', 'bandwidth': 57.578851736563706, 'diag': False, 'norm_p': 1.4050681380716323, 'reg': 0.002443825456653774}
Best macro F1: 0.796167149321403


In [8]:
model, pipeline, fit_time, train_metrics, val_metrics = fit_best_xrfm(
    X,
    y,
    best_trial,
    processor,
    n_num_cols,
    task=cfg["task"],
    device=device,
)

print(f"Training time: {fit_time:.2f}s")
print(
    f"Train Accuracy: {train_metrics['accuracy']:.4f}  AUC-ROC: {train_metrics['auc_roc']:.4f}"
)
print(
    f"Val Accuracy: {val_metrics['accuracy']:.4f}  AUC-ROC: {val_metrics['auc_roc']:.4f}"
)

joblib.dump(model, MODELS_DIR / data_name / "xrfm.joblib")
joblib.dump(pipeline, MODELS_DIR / data_name / "xrfm_pipeline.joblib")

None
Fitting xRFM with 1 trees and 0 iterations per tree


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]

Fitting RFM with ntrain: 12800, d: 70, and nval: 3200
Using expensive batch size
Optimal M batch size: 398
Using SVD
Time taken for round 0: 2.1988418102264404 seconds
Using expensive batch size
Optimal M batch size: 398
Using SVD
Time taken for round 1: 2.137434959411621 seconds
Using expensive batch size
Optimal M batch size: 398
Using SVD
Time taken for round 2: 2.1404078006744385 seconds
Using expensive batch size
Optimal M batch size: 398
Using SVD
Time taken for round 3: 2.140713930130005 seconds
Using expensive batch size
Optimal M batch size: 398
Using SVD
Time taken for round 4: 2.0965147018432617 seconds


Building trees:   0%|          | 0/1 [00:11<?, ?it/s]

Tree has no split, stopping training
Using hard routing for tree prediction


Using hard routing for tree prediction
Training time: 11.61s
Train Accuracy: 1.0000  AUC-ROC: 1.0000
Val Accuracy: 0.8859  AUC-ROC: 0.9743


['/home/shado/college_project/comp9417/9417GroupProject/models/sleep_health/xrfm_pipeline.joblib']

In [9]:
model = joblib.load(MODELS_DIR / data_name / "xrfm.joblib")
pipeline = joblib.load(MODELS_DIR / data_name / "xrfm_pipeline.joblib")

In [10]:
X_test = test_df.drop(columns=metadata["target_col"])
y_test = le.transform(test_df[metadata["target_col"]]).astype(np.int32)

X_test = pipeline.transform(X_test)
t0 = time.perf_counter()
y_proba = model.predict_proba(X_test)
infer_time = (time.perf_counter() - t0) / len(X_test)

Using hard routing for tree prediction


In [11]:
results = evaluate_classification(y_test, y_proba)
print(f"Accuracy:  {results['accuracy']:.4f}")
print(f"AUC-ROC:   {results['auc_roc']:.4f}")
print(f"Inference time per sample: {infer_time * 1e6:.2f}µs")

Accuracy:  0.8905
AUC-ROC:   0.9756
Inference time per sample: 49.71µs


In [12]:
output = {
    **results,
    "train_metrics": train_metrics,
    "val_metrics": val_metrics,
    "fit_time_s": fit_time,
    "infer_time_per_sample_s": infer_time,
    "best_params": {
        "tuning_metric": model.tuning_metric,
        "split_method": model.split_method,
        "max_leaf_size": model.max_leaf_size,
        **model.rfm_params,
    },
}

results_dir = RESULTS_BENCHMARK / "sleep_health"
with open(results_dir / "xrfm.json", "w") as f:
    json.dump(output, f, indent=4)

plot_roc_curve(
    y_test,
    y_proba,
    save_path=results_dir / "roc_curve_xrfm.png",
    class_names=list(le.classes_),
)
plot_confusion_matrix(
    y_test,
    y_proba,
    save_path=results_dir / "confusion_matrix_xrfm.png",
    class_names=list(le.classes_),
)
save_classification_report(
    y_test,
    y_proba,
    save_path=results_dir / "classification_report_xrfm.json",
    class_names=list(le.classes_),
)

print(f"Results saved to {results_dir / 'xrfm.json'}")

Results saved to /home/shado/college_project/comp9417/9417GroupProject/results/sleep_health/xrfm.json
